# Autocomplete and Autocorrect Data Analytics
**OIBSIP · Data Analytics · Level 2 · Task 5**

This notebook builds, tests and compares:
1. **Autocomplete** - frequency-based bigram / trigram models that predict the next word (and complete a typed prefix).
2. **Autocorrect** - edit-distance spelling correctors (Norvig-style, `pyspellchecker`, and a context-aware variant).

**Pipeline:** corpus -> preprocessing -> frequency analysis -> autocomplete -> autocorrect -> metrics -> comparison -> limitations.

## 1. Setup

In [ ]:
import re, random, time, string, urllib.request
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import nltk
    HAS_NLTK = True
except ImportError:
    HAS_NLTK = False

try:
    from spellchecker import SpellChecker
    HAS_PYSPELL = True
except ImportError:
    HAS_PYSPELL = False

SEED = 42
random.seed(SEED); np.random.seed(SEED)
sns.set_theme(style="whitegrid")
print("NLTK:", HAS_NLTK, "| pyspellchecker:", HAS_PYSPELL)

## 2. Collect the Text Corpus
I use public-domain books from **Project Gutenberg** (a large, clean, free corpus). If the download fails (no internet),
place any `.txt` files in a folder called `data/` and they will be used instead.

In [ ]:
import os, glob
BOOKS = {
    "Pride and Prejudice": "https://www.gutenberg.org/cache/epub/1342/pg1342.txt",
    "Adventures of Sherlock Holmes": "https://www.gutenberg.org/cache/epub/1661/pg1661.txt",
    "Alice in Wonderland": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "Frankenstein": "https://www.gutenberg.org/cache/epub/84/pg84.txt",
}
os.makedirs("data", exist_ok=True)

def strip_gutenberg(text):
    # remove the licence header/footer that wraps every Gutenberg book
    st = re.search(r"\*\*\* ?START OF (THE|THIS) PROJECT GUTENBERG.*?\*\*\*", text, re.S)
    if st: text = text[st.end():]
    en = re.search(r"\*\*\* ?END OF (THE|THIS) PROJECT GUTENBERG", text)
    if en: text = text[:en.start()]
    return text

texts = {}
for name, url in BOOKS.items():
    path = f"data/{re.sub(r'[^a-z]+','_',name.lower())}.txt"
    try:
        if not os.path.exists(path):
            raw = urllib.request.urlopen(url, timeout=20).read().decode("utf-8", "ignore")
            open(path, "w", encoding="utf-8").write(raw)
        texts[name] = strip_gutenberg(open(path, encoding="utf-8").read())
    except Exception as ex:
        print("Could not download", name, "->", type(ex).__name__)

if not texts:                                   # fallback: any local .txt files
    for p in glob.glob("data/*.txt"):
        texts[os.path.basename(p)] = open(p, encoding="utf-8", errors="ignore").read()

corpus = "\n".join(texts.values())
print(f"Books loaded: {len(texts)} | characters: {len(corpus):,}")

## 3. NLP Preprocessing
Steps: normalise quotes -> sentence split -> **lowercase** -> **punctuation removal** -> **tokenisation**.

**Stopword removal** is applied only for the *frequency analysis* chart. It is deliberately **not** applied to the
autocomplete/autocorrect models, because next-word prediction must see words like "of", "the", "to" - they are the most common words people actually type.

In [ ]:
def get_stopwords():
    if HAS_NLTK:
        try:
            nltk.download("stopwords", quiet=True)
            from nltk.corpus import stopwords
            return set(stopwords.words("english"))
        except Exception:
            pass
    return set("a about above after again all am an and any are as at be because been before being below between both but by "
               "can did do does doing down during each few for from further had has have having he her here hers him his how i if in "
               "into is it its me more most my no nor not of off on once only or other our out over own s same she should so some such "
               "than that the their them then there these they this those through to too under until up very was we were what when where "
               "which while who whom why will with you your yours t d ll m o re ve".split())
STOP = get_stopwords()

def tokenize(sentence):
    sentence = sentence.replace("\u2019", "'").replace("\u2018", "'").lower()
    toks = re.findall(r"[a-z']+", sentence)            # strips punctuation & digits
    toks = [t.strip("'") for t in toks]
    return [t for t in toks if t]

def split_sentences(text):
    text = re.sub(r"\s+", " ", text)
    return [s for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

sentences = [tokenize(s) for s in split_sentences(corpus)]
sentences = [s for s in sentences if len(s) >= 3]

# 80/20 split BY SENTENCE so test data is unseen
random.shuffle(sentences)
cut = int(0.8 * len(sentences))
train_sents, test_sents = sentences[:cut], sentences[cut:]
train_tokens = [w for s in train_sents for w in s]

print(f"Sentences: {len(sentences):,} (train {len(train_sents):,} / test {len(test_sents):,})")
print(f"Training tokens: {len(train_tokens):,} | vocabulary: {len(set(train_tokens)):,}")
print("Sample:", train_sents[0][:15])

## 4. Word-Frequency Analysis

In [ ]:
freq = Counter(train_tokens)
freq_nostop = Counter({w: c for w, c in freq.items() if w not in STOP and len(w) > 1})
top20 = pd.DataFrame(freq_nostop.most_common(20), columns=["word", "count"])

plt.figure(figsize=(10, 6))
sns.barplot(data=top20, y="word", x="count", color="#3b82f6")
plt.title("Top 20 Most Frequent Words (stopwords removed)")
plt.xlabel("Frequency"); plt.ylabel("")
plt.tight_layout(); plt.savefig("top20_words.png", dpi=150); plt.show()

**Observation:** The distribution follows *Zipf's law* - a handful of words account for a large share of all usage while most words are rare.
This is why frequency-based prediction works well for common words but struggles with the long tail.

## 5. Autocomplete
### 5.1 N-gram models
* **Bigram:** P(next | previous word)
* **Trigram:** P(next | previous two words)
* **Trigram with backoff:** try trigram -> bigram -> unigram so a prediction is *always* returned.

In [ ]:
START = "<s>"
unigram = Counter(train_tokens)
bigram, trigram = defaultdict(Counter), defaultdict(Counter)
for s in train_sents:
    padded = [START, START] + s
    for i in range(2, len(padded)):
        bigram[padded[i-1]][padded[i]] += 1
        trigram[(padded[i-2], padded[i-1])][padded[i]] += 1

top_unigrams = [w for w, _ in unigram.most_common(50)]

def predict_bigram(ctx, k=3):
    prev = ctx[-1] if ctx else START
    return [w for w, _ in bigram[prev].most_common(k)] if prev in bigram else []

def predict_trigram(ctx, k=3):
    c = ([START, START] + list(ctx))[-2:]
    key = tuple(c)
    return [w for w, _ in trigram[key].most_common(k)] if key in trigram else []

def predict_backoff(ctx, k=3):
    out = predict_trigram(ctx, k)
    for src in (predict_bigram(ctx, k), top_unigrams):
        for w in src:
            if len(out) >= k: break
            if w not in out: out.append(w)
    return out[:k]

def complete_prefix(prefix, ctx=(), k=3):
    # word completion: finish a partially typed word, ranked by context first, then global frequency
    prefix = prefix.lower()
    ranked = [w for w in predict_backoff(ctx, 200) if w.startswith(prefix)]
    ranked += [w for w, _ in unigram.most_common() if w.startswith(prefix) and w not in ranked]
    return ranked[:k]

### 5.2 Test on 10+ input prefixes (top-3 predictions)

In [ ]:
test_prefixes = ["i am", "it was", "of the", "in the", "he said", "she was not", "i do not", "there is", "what is the", "you must"]
rows = []
for p in test_prefixes:
    ctx = p.split()
    rows.append({"input": p, "bigram": predict_bigram(ctx), "trigram": predict_trigram(ctx), "backoff (top 3)": predict_backoff(ctx)})
display(pd.DataFrame(rows))

print("\nPrefix completion examples")
for pre, ctx in [("th", ["of"]), ("wor", ["the"]), ("beau", ["a"]), ("con", ["i"]), ("sh", [])]:
    print(f"  '{pre}' after {ctx} ->", complete_prefix(pre, ctx))

### 5.3 Autocomplete metrics (on unseen test sentences)
For each position in a test sentence the model sees the previous words and must predict the true next word.

| Metric | Definition |
|---|---|
| **Top-1 / Top-3 accuracy** | true word is the 1st / among the 3 suggestions (over *all* test cases) |
| **Coverage** | share of cases where the model returned any suggestion |
| **Precision@3** | hits ÷ cases where suggestions were shown (quality when it speaks) |
| **Recall@3** | hits ÷ all test cases (how much of real typing it can predict) |

In [ ]:
def eval_autocomplete(fn, sents, k=3):
    total = answered = hit1 = hit3 = 0
    for s in sents:
        for i in range(len(s)):
            ctx, truth = s[:i], s[i]
            preds = fn(ctx, k)
            total += 1
            if preds:
                answered += 1
                hit1 += preds[0] == truth
                hit3 += truth in preds
    return {"Coverage": answered/total, "Top-1 acc": hit1/total, "Top-3 acc (Recall@3)": hit3/total,
            "Precision@3": hit3/answered if answered else 0, "Test cases": total}

eval_sents = test_sents[:3000]
ac_results = pd.DataFrame({name: eval_autocomplete(fn, eval_sents) for name, fn in
                           [("Bigram", predict_bigram), ("Trigram", predict_trigram), ("Trigram + backoff", predict_backoff)]}).T
display(ac_results.style.format({c: "{:.2%}" for c in ac_results.columns if c != "Test cases"}))

ac_results[["Top-1 acc", "Top-3 acc (Recall@3)", "Precision@3", "Coverage"]].plot(kind="bar", figsize=(9, 5), rot=0)
plt.title("Autocomplete model comparison"); plt.ylabel("Score"); plt.tight_layout()
plt.savefig("autocomplete_comparison.png", dpi=150); plt.show()

**Observation:** The trigram is *precise* when it has seen the context but has low coverage (sparse data), the bigram covers more but is less precise,
and **backoff** combines both - this is the same trade-off real keyboards face.

## 6. Autocorrect
### 6.1 Approaches compared
1. **Norvig edit-distance corrector** - generates every string within edit distance 1 (then 2) of the typo, keeps real words, picks the most frequent.
2. **`pyspellchecker`** - library implementing Levenshtein distance against its own word-frequency dictionary.
3. **Context-aware corrector** - same candidates as (1) but re-ranked using the *previous word* (bigram probability), so "their/there" style ties are broken by context.

Edit distance (Levenshtein) = minimum number of insertions, deletions, substitutions (and transpositions in Damerau variant) to turn one word into another.

In [ ]:
LETTERS = string.ascii_lowercase
VOCAB = set(unigram)
N_TOKENS = sum(unigram.values())
V = len(VOCAB)

def levenshtein(a, b):
    # classic dynamic-programming edit distance (used for demonstration / analysis)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def edits1(word):
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes = [L + R[1:] for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
    replaces = [L + c + R[1:] for L, R in splits if R for c in LETTERS]
    inserts = [L + c + R for L, R in splits for c in LETTERS]
    return set(deletes + transposes + replaces + inserts)

def known(words): return {w for w in words if w in VOCAB}

def candidates(word):
    if word in VOCAB: return {word}
    e1 = known(edits1(word))
    if e1: return e1
    e2 = known(e2w for e1w in edits1(word) for e2w in edits1(e1w))
    return e2 or {word}

def correct_norvig(word, prev=None):
    return max(candidates(word), key=lambda w: unigram[w])

def correct_context(word, prev=None):
    cands = candidates(word)
    if prev is None or len(cands) == 1:
        return max(cands, key=lambda w: unigram[w])
    # score = P(w) * P(w | prev) with add-one smoothing
    def score(w):
        p_uni = (unigram[w] + 1) / (N_TOKENS + V)
        p_bi = (bigram[prev][w] + 1) / (sum(bigram[prev].values()) + V) if prev in bigram else 1 / V
        return p_uni * p_bi
    return max(cands, key=score)

if HAS_PYSPELL:
    _spell = SpellChecker()
    def correct_pyspell(word, prev=None):
        return _spell.correction(word) or word

print("levenshtein('speling','spelling') =", levenshtein("speling", "spelling"))
print("norvig  : speling ->", correct_norvig("speling"), "| korrect ->", correct_norvig("korrect"))

### 6.2 Test on 20 deliberately misspelled words

In [ ]:
manual_tests = {
    "recieve": "receive", "beleive": "believe", "wich": "which", "thier": "their", "becuase": "because",
    "freind": "friend", "tomorow": "tomorrow", "definately": "definitely", "occured": "occurred", "seperate": "separate",
    "adress": "address", "begining": "beginning", "untill": "until", "writting": "writing", "gerat": "great",
    "wonderfull": "wonderful", "mornig": "morning", "peple": "people", "beutiful": "beautiful", "diffrent": "different",
}
methods = {"Norvig (edit distance)": correct_norvig, "Context-aware": correct_context}
if HAS_PYSPELL: methods["pyspellchecker"] = correct_pyspell

rows = []
for typo, truth in manual_tests.items():
    r = {"misspelled": typo, "expected": truth}
    for name, fn in methods.items():
        out = fn(typo)
        r[name] = ("OK " if out == truth else "X  ") + out
    rows.append(r)
manual_df = pd.DataFrame(rows)
display(manual_df)
for name in methods:
    print(f"{name:25s} accuracy on 20 words: {(manual_df[name].str.startswith('OK')).mean():.0%}")

*(Words not present in the corpus vocabulary, e.g. archaic spellings or modern words, cannot be corrected by a corpus-trained model - an important limitation discussed in Section 9.)*

### 6.3 Large-scale evaluation with synthetic typos
Hand-picked words are too few for reliable statistics, so I generate typos automatically from real **held-out test words** (random deletion, insertion, substitution or transposition), keeping the previous word as context.

In [ ]:
def make_typo(word):
    for _ in range(10):
        i = random.randrange(len(word))
        op = random.choice(["delete", "insert", "replace", "swap"])
        if op == "delete":   t = word[:i] + word[i+1:]
        elif op == "insert": t = word[:i] + random.choice(LETTERS) + word[i:]
        elif op == "replace":t = word[:i] + random.choice(LETTERS) + word[i+1:]
        else:
            if len(word) < 2: continue
            i = min(i, len(word) - 2); t = word[:i] + word[i+1] + word[i] + word[i+2:]
        if t != word and t not in VOCAB and len(t) > 1:   # must be a true non-word error
            return t
    return None

cases = []
for s in test_sents:
    for i, w in enumerate(s):
        if len(w) >= 4 and w in VOCAB and w.isalpha():
            t = make_typo(w)
            if t: cases.append((t, w, s[i-1] if i > 0 else None))
random.shuffle(cases)
cases = cases[:400]
print("Evaluation cases:", len(cases))

def evaluate_corrector(fn, cases):
    t0 = time.time()
    fixed = [fn(t, p) for t, w, p in cases]               # misspelled -> corrected?
    clean = [fn(w, p) for t, w, p in cases]               # correct words -> must stay unchanged
    secs = time.time() - t0
    N = len(cases)
    tp = sum(f == w for f, (t, w, p) in zip(fixed, cases))
    changed_wrong_on_typos = sum((f != t) and (f != w) for f, (t, w, p) in zip(fixed, cases))
    false_changes = sum(c != w for c, (t, w, p) in zip(clean, cases))
    total_changes = tp + changed_wrong_on_typos + false_changes
    return {
        "Accuracy (typos fixed)": tp / N,
        "Precision": tp / total_changes if total_changes else 0,
        "Recall": tp / N,
        "False-correction rate": false_changes / N,
        "ms / word": 1000 * secs / (2 * N),
        "_tp": tp, "_fixed": fixed, "_clean": clean,
    }

ac_eval = {name: evaluate_corrector(fn, cases) for name, fn in methods.items()}
summary = pd.DataFrame({n: {k: v for k, v in r.items() if not k.startswith("_")} for n, r in ac_eval.items()}).T
summary["F1"] = 2 * summary["Precision"] * summary["Recall"] / (summary["Precision"] + summary["Recall"])
display(summary.style.format({c: "{:.2%}" for c in summary.columns if c != "ms / word"} | {"ms / word": "{:.2f}"}))

**Metric definitions** - *Recall* = typos correctly fixed ÷ all typos. *Precision* = correct fixes ÷ all words the system changed
(including wrongly "fixing" a word that was already right). *False-correction rate* = already-correct words that were wrongly changed.

### 6.4 Confusion matrix - correct vs. incorrect, before and after autocorrect

In [ ]:
def before_after_matrix(res, cases):
    N = len(cases)
    typo_ok = res["_tp"]
    clean_ok = sum(c == w for c, (t, w, p) in zip(res["_clean"], cases))
    # rows = state BEFORE autocorrect, columns = state AFTER
    return pd.DataFrame([[typo_ok, N - typo_ok], [clean_ok, N - clean_ok]],
                        index=["Before: misspelled", "Before: correct"],
                        columns=["After: correct", "After: incorrect"])

fig, axes = plt.subplots(1, len(methods), figsize=(5.5 * len(methods), 4.5))
axes = np.atleast_1d(axes)
for ax, (name, res) in zip(axes, ac_eval.items()):
    sns.heatmap(before_after_matrix(res, cases), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(name)
plt.tight_layout(); plt.savefig("autocorrect_confusion_matrix.png", dpi=150); plt.show()

## 7. Algorithm Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
summary[["Accuracy (typos fixed)", "Precision", "Recall", "F1"]].plot(kind="bar", ax=ax, rot=0)
ax.set_title("Autocorrect: algorithm comparison"); ax.set_ylabel("Score"); ax.set_ylim(0, 1)
plt.tight_layout(); plt.savefig("autocorrect_comparison.png", dpi=150); plt.show()

# Edit-distance analysis: which error types / word lengths are hardest?
analysis = pd.DataFrame({
    "typo": [t for t, w, p in cases], "truth": [w for t, w, p in cases],
    "fixed": ac_eval["Norvig (edit distance)"]["_fixed"]})
analysis["correct"] = analysis.fixed == analysis.truth
analysis["edit_dist"] = [levenshtein(t, w) for t, w in zip(analysis.typo, analysis.truth)]
analysis["word_len"] = analysis.truth.str.len()
print("Accuracy by word length (Norvig):")
display(analysis.groupby(pd.cut(analysis.word_len, [3, 5, 7, 9, 30])).correct.mean().to_frame("accuracy").style.format("{:.1%}"))
print("Examples the Norvig model got wrong:")
display(analysis[~analysis.correct].head(10)[["typo", "truth", "fixed"]])

## 8. Key Findings
*(Fill in with your actual numbers after running the notebook.)*

* **Autocomplete:** trigram + backoff achieved Top-3 accuracy of ___ % vs. ___ % (bigram only).
* **Autocorrect:** the best method was ___ with ___ % accuracy and ___ % false-correction rate.
* **Context helps** when several candidates are equally close in edit distance, but costs more computation per word.
* Frequency-based methods favour common words, so rare-but-correct words are sometimes "corrected" wrongly.

## 9. Limitations vs. Production Systems (e.g. Google Gboard)
| Area | This project | Production keyboard |
|---|---|---|
| **Model** | Count-based n-grams (max 2 words of context) | Neural language models (LSTM / Transformer) that use long context and understand meaning |
| **Data** | A few public-domain novels - 19th-century vocabulary | Billions of modern messages, slang, names, emojis, multiple languages |
| **Personalisation** | None | Learns your own vocabulary, contacts and writing style on-device (federated learning) |
| **Error model** | Uniform random typos | Models *keyboard geometry* (adjacent keys), fat-finger touch probabilities, swipe typing |
| **Real-word errors** | Cannot catch "their/there" (only non-word errors) | Context-aware detection of valid-but-wrong words |
| **Out-of-vocabulary words** | Unknown words cannot be corrected | Sub-word models and character-level fallbacks |
| **Speed / size** | Brute-force edit generation (slow for edit distance 2) | Optimised tries/FSTs, quantised models, < 10 ms latency, runs on-device |
| **Evaluation** | Offline accuracy | Live A/B tests, keystroke savings, user satisfaction |

**Possible improvements:** Kneser-Ney smoothing, a character-level keyboard-distance error model, a trie/BK-tree for faster lookup, a small neural LM, and a much larger modern corpus.

## 10. Conclusion
Frequency-based n-gram models give a solid, explainable baseline for autocomplete, and edit-distance correction with frequency (and optional context) ranking fixes most single-typo errors.
Both approaches show clear trade-offs between **coverage, precision, and speed**, and they highlight why modern systems moved to neural language models.